# Klike — Recommendations Engine interativo

Este notebook apresenta o protótipo do desafio Millai / Klike. Informe uma campanha e clique em **Gerar recomendações**. O painel substitui o resultado anterior a cada execução.

> **Para quem deseja apenas testar sem modificar notebooks Git:** prefira [a interface Streamlit](../app.py), que funciona no navegador e não grava saídas em arquivos `.ipynb`. Execute `python -m streamlit run app.py` na raiz do repositório.

O Ridge usa apenas características do criativo; associações contextuais são calculadas sobre o treinamento. Previsões e comparações não são estimativas causais.


In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "klike_challenge_dataset.csv").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data" / "klike_challenge_dataset.csv").is_file():
    raise FileNotFoundError("Execute a partir da raiz do projeto ou de notebooks/.")
sys.path.insert(0, str(PROJECT_ROOT))
from src.recommendations_engine import RecommendationsEngine
engine = RecommendationsEngine(PROJECT_ROOT)
print(f"Ridge carregado. Referência contextual: {len(engine.df_train)} campanhas de treinamento.")


## Configure a campanha

As dez entradas abaixo correspondem às features do Ridge reduzido. As opções categóricas vêm do conjunto de treinamento. A análise pode levar alguns segundos devido às reamostragens bootstrap.


In [ ]:
import ipywidgets as widgets
from html import escape
from IPython.display import display

style = {"description_width": "155px"}
wide = widgets.Layout(width="390px")
campaign_id = widgets.Text(value="CAMPANHA-NOVA", description="Identificador", style=style, layout=wide)
platform = widgets.Dropdown(options=engine.categorias_validas["platform"], value="Meta", description="Plataforma", style=style, layout=wide)
category = widgets.Dropdown(options=engine.categorias_validas["category"], value="E-commerce", description="Categoria", style=style, layout=wide)
video_duration_s = widgets.BoundedIntText(value=16, min=1, max=10000, description="Duração (s)", style=style, layout=wide)
video_format = widgets.Dropdown(options=engine.categorias_validas["format"], value="vertical", description="Formato", style=style, layout=wide)
text_density = widgets.Dropdown(options=engine.categorias_validas["text_density"], value="low", description="Densidade de texto", style=style, layout=wide)
is_retargeting = widgets.Checkbox(value=False, description="Retargeting", indent=False)
has_subtitle = widgets.Checkbox(value=False, description="Legendas", indent=False)
has_cta = widgets.Checkbox(value=False, description="CTA", indent=False)
has_hook = widgets.Checkbox(value=False, description="Hook", indent=False)
has_face = widgets.Checkbox(value=False, description="Rosto humano", indent=False)

button = widgets.Button(description="Gerar recomendações", button_style="success", icon="check", layout=widgets.Layout(width="250px"))
# HTML.value substitui a visualização, ao contrário de Output, que pode acumular eventos no VS Code.
painel = widgets.HTML(
    value="<div style='padding:14px;border:1px solid #ddd;border-radius:8px'>Preencha os campos e clique em <b>Gerar recomendações</b>.</div>",
    layout=widgets.Layout(width="100%")
)
form = widgets.VBox([
    widgets.HTML("<h3>Configurações</h3>"),
    campaign_id, platform, category, video_duration_s, video_format, text_density,
    widgets.HTML("<h3>Características do criativo</h3>"),
    widgets.HBox([has_hook, has_face]), widgets.HBox([has_subtitle, has_cta]),
    is_retargeting, button, painel
])
display(form)


In [ ]:
def on_recommend_click(_):
    button.disabled = True
    button.description = "Calculando..."
    painel.value = "<p>Calculando previsões e associações contextuais...</p>"
    try:
        campanha = {
            "campaign_id": campaign_id.value.strip() or "CAMPANHA-NOVA",
            "platform": platform.value,
            "category": category.value,
            "is_retargeting": is_retargeting.value,
            "video_duration_s": video_duration_s.value,
            "format": video_format.value,
            "has_subtitle": has_subtitle.value,
            "has_cta": has_cta.value,
            "has_hook": has_hook.value,
            "has_face": has_face.value,
            "text_density": text_density.value,
        }
        result = engine.recomendar_campanha(
            campanha, max_recomendacoes=3, delta_minimo=1.0, n_bootstrap=2000
        )
        blocos = [
            "<div style='border:1px solid #ddd;border-radius:10px;padding:18px'>",
            f"<h3>Resultado · {escape(str(result['campaign_id']))}</h3>",
            (
                "<p><b>Klike Score previsto:</b> "
                f"{result['score_original']:.2f} pontos &nbsp; · &nbsp; "
                f"<b>Cenários:</b> {result['total_cenarios']} &nbsp; · &nbsp; "
                f"<b>Recomendações elegíveis:</b> {result['total_recomendacoes']}</p>"
            ),
        ]
        if result["recomendacoes"]:
            for r in result["recomendacoes"]:
                acao = engine.descrever_acao(
                    r["feature"], r["valor_original"], r["valor_proposto"]
                )
                blocos.append(
                    "<section style='border-top:1px solid #ddd;padding:12px 0'>"
                    f"<h4>{r['prioridade']}ª — {escape(acao)}</h4>"
                    f"<p><b>Diferença prevista: {r['delta_ridge']:+.2f} pontos</b></p>"
                    f"<p>{escape(r['texto'])}</p>"
                    "<p style='font-size:12px'>"
                    f"Grupos de referência: {r['n_original']} e {r['n_proposto']} campanhas"
                    "</p></section>"
                )
        else:
            blocos.append("<p>Nenhuma alteração atendeu aos critérios de elegibilidade.</p>")
        if result["total_recomendacoes"] < 3:
            blocos.append(
                "<p><i>Há menos de três oportunidades elegíveis. "
                "Nenhum ganho é inventado para completar a lista.</i></p>"
            )
        if not result["avaliacao"].empty:
            colunas = ["Feature", "Original", "Proposto", "Delta Ridge", "Status decisão"]
            diagnostico = result["avaliacao"][colunas].round(3).to_html(
                index=False, escape=True, border=0
            )
            blocos.append("<details><summary>Diagnóstico dos cenários</summary>")
            blocos.append(diagnostico)
            blocos.append("</details>")
        blocos.append(
            "<p style='font-size:12px'>Previsões e associações observacionais "
            "não representam efeitos causais.</p></div>"
        )
        # Uma única atualização: não há display(), print() ou Output por clique.
        painel.value = "".join(blocos)
    except (ValueError, TypeError, FileNotFoundError, KeyError) as exc:
        painel.value = (
            "<div style='padding:12px;border:1px solid #cc7777'>"
            f"Não foi possível avaliar a campanha: {escape(str(exc))}</div>"
        )
    finally:
        button.disabled = False
        button.description = "Gerar recomendações"

# Evitar múltiplos callbacks caso apenas esta célula seja executada novamente.
callback_anterior = getattr(button, "_klike_callback", None)
if callback_anterior is not None:
    button.on_click(callback_anterior, remove=True)
button.on_click(on_recommend_click)
button._klike_callback = on_recommend_click


## Notas de interpretação

A diferença prevista no Klike Score resulta da alteração isolada de uma característica, mantendo as demais fixas no Ridge reduzido. As evidências contextuais usam somente o treino e intervalos bootstrap exploratórios. Os limiares de 10 observações por grupo e delta mínimo de 1 ponto são heurísticos, não garantias de confiabilidade ou efeito causal.

**Metodologia, validações e conclusões:** veja [parte-3.ipynb](parte-3.ipynb).
